# Autonomous Data Analyst Agent

Ask questions about a CSV in plain English, get back SQL, a verified answer, and a chart — all running locally through Ollama. No API key, no data leaving the machine.

**Stack:** Python, Pandas, SQLite, Ollama (`qwen2.5-coder`), Plotly

## What it does
- Loads any local CSV into SQLite
- Translates the question into a SQL query with a local LLM
- Executes it — if it errors, the error goes straight back to the model to self-correct
- Runs a second, independent LLM pass to verify the SQL/result actually answers the question, and retries with feedback if it doesn't
- Refuses to execute anything but read-only `SELECT`/`WITH` queries
- Auto-picks a chart type and writes a short narrative grounded only in the rows actually returned

## Setup
1. Install [Ollama](https://ollama.com), then pull the model: `ollama pull qwen2.5-coder`
2. Start the server in a separate terminal: `ollama serve`
3. `pip install ollama pandas plotly`
4. Run the cells top to bottom. In the data-layer cell, set `CSV_PATH` to any CSV on your machine — that's the only line you need to touch to point this at a new dataset.

## Design notes (for whoever's reading the code)
- I originally tried Ollama's native tool-calling for the run_sql / final_answer / clarify actions, but `qwen2.5-coder` wasn't triggering it reliably on my setup, so I switched to `format="json"` with the action schema written into the system prompt instead. Less elegant, much more reliable.
- The biggest real bug I hit: a zero-row query result isn't the same thing as "no data exists," but the first version of this treated them identically and reported "no data found" way too often. Almost every case was actually a wrong filter value (case/spelling) or an unquoted column name with a space in it breaking the SQL. Fixed by sanitizing column names on load, giving the model real example values per column instead of just names, and treating an empty result as an unverified answer that gets one more retry before the agent is allowed to conclude there's genuinely no match.

## Limitations
- Single-table reasoning is solid; multi-table joins are hit or miss on a model this small
- Everything runs locally, so latency is a few seconds to tens of seconds per question — this isn't tuned for production QPS
- The eval set below is a smoke test I run after touching the prompts, not a full benchmark


In [ ]:
import os
import re
import json
import time
import sqlite3
import pandas as pd
import ollama
import plotly.express as px

MODEL = "qwen2.5-coder"       # ollama pull qwen2.5-coder
DB_PATH = "data.db"
MAX_SQL_FIX_ATTEMPTS = 2      # retries inside one turn for a query that errors out
MAX_VERIFY_RETRIES = 2        # retries for a query that runs but looks wrong / empty

# most of my early debugging sessions turned out to just be "ollama serve" not running,
# so check that loudly before anything else instead of failing 5 cells from now
try:
    ping = ollama.chat(model=MODEL, messages=[{"role": "user", "content": "reply with just: ready"}])
    print("Ollama OK:", ping["message"]["content"].strip())
except Exception as e:
    print("Can't reach Ollama — is `ollama serve` running and is the model pulled? ->", e)

con = sqlite3.connect(DB_PATH)
print("SQLite OK:", DB_PATH)

In [ ]:
def _sanitize_columns(df):
    # Kaggle-style headers ("Product Name", "Sales ($)") break unquoted SQL — normalize
    # once at load time instead of fighting it in every generated query.
    clean = df.columns.str.strip().str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True).str.strip("_")
    df.columns = [c if c and not c[0].isdigit() else f"col_{c}" for c in clean]
    return df


def load_data(csv_path: str, table_name: str = "data"):
    """Loads a local CSV into SQLite. Point this at any file on your machine — swap
    datasets by re-running with a new path, the old table just gets replaced."""
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Can't find '{csv_path}' — check the path and try again.")

    try:
        df = pd.read_csv(csv_path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding="latin1")  # a lot of older Kaggle exports aren't UTF-8

    df = _sanitize_columns(df)
    df.to_sql(table_name, con, if_exists="replace", index=False)
    print(f"Loaded {len(df)} rows x {len(df.columns)} cols into '{table_name}'")
    return table_name


def get_schema():
    """Column names/types plus a few real example values per column. The examples matter —
    without them the model guesses at filter values ('electronics' vs 'Electronics') and
    quietly comes back with zero rows instead of the right answer."""
    tables = con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    lines = []
    for (table_name,) in tables:
        columns = con.execute(f"PRAGMA table_info({table_name})").fetchall()
        sample = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 25", con)
        col_lines = []
        for col in columns:
            name = col[1]
            examples = sample[name].dropna().unique().tolist()[:3] if name in sample else []
            col_lines.append(f"  - {name} ({col[2]}), e.g. {examples}")
        lines.append(f"Table '{table_name}':\n" + "\n".join(col_lines))
    return "\n".join(lines)


READ_ONLY = re.compile(r"^\s*(SELECT|WITH)\b", re.IGNORECASE)
BLOCKED = re.compile(r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|ATTACH|PRAGMA|REPLACE|CREATE)\b", re.IGNORECASE)


def run_sql(query: str):
    """Executes SQL, returns (df, error) — never raises, the agent loop depends on that.
    Also the one chokepoint every generated query passes through, so it's where read-only
    gets enforced: the agent should never touch the DB beyond SELECTing from it."""
    if not READ_ONLY.match(query) or BLOCKED.search(query):
        return None, "Only a single SELECT/WITH statement is allowed."
    try:
        return pd.read_sql_query(query, con), None
    except Exception as e:
        return None, str(e)


# ---- point this at any CSV on your machine ----
CSV_PATH = r"PUT_YOUR_CSV_PATH_HERE.csv"
load_data(CSV_PATH, table_name="sales")
print(get_schema())

In [ ]:
def call_llm(messages):
    # plain text completion — only used for free-text narrative writing, not for decisions
    return ollama.chat(model=MODEL, messages=messages)["message"]["content"]


def call_llm_json(messages):
    # forces strict JSON — every decision the agent acts on goes through this
    content = ollama.chat(model=MODEL, messages=messages, format="json")["message"]["content"]
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        # model occasionally wraps JSON in prose despite the format flag — treat as
        # "done" rather than crashing the loop
        return {"action": "final_answer"}


def agent_step(question: str, schema: str, feedback: str = None):
    """One reasoning pass: model sees the schema + question and decides to run SQL, ask
    for clarification, or say it's done. Query errors get fed straight back as the next
    message — that's the self-fixing part, and it usually corrects a typo'd column name
    or a missing GROUP BY in one try."""
    system_prompt = f"""You are a data analyst. You answer questions by writing SQLite SELECT queries.

Database schema:
{schema}

Respond with ONLY a JSON object, no prose, no markdown fences. Exactly one of:
{{"action": "run_sql", "query": "SELECT ..."}}
{{"action": "final_answer"}}
{{"action": "clarify", "clarifying_question": "..."}}

Rules:
- Only SELECT/WITH statements, only tables/columns from the schema above.
- Match filter values exactly as shown in the schema examples (case, spelling, punctuation).
- If a query errors, read the error and fix it — don't repeat the same mistake.
- Call final_answer once you have the data needed to answer the question."""

    if feedback:
        system_prompt += f"\n\nA previous attempt at this exact question was wrong: {feedback}\nTry a genuinely different query."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    last_sql, last_df, sql_error = None, None, None

    for _ in range(MAX_SQL_FIX_ATTEMPTS + 1):
        parsed = call_llm_json(messages)
        messages.append({"role": "assistant", "content": json.dumps(parsed)})
        action = parsed.get("action")

        if action == "run_sql":
            query = parsed.get("query", "")
            df, error = run_sql(query)
            last_sql = query
            if error:
                sql_error, last_df = error, None
                messages.append({"role": "user", "content": f"Query failed: {error}. Fix it and reply with JSON again."})
                continue
            sql_error, last_df = None, df
            preview = df.head(20).to_json(orient="records")
            messages.append({"role": "user", "content": f"Result ({len(df)} rows): {preview}\nCall final_answer if this answers the question, or run_sql again if you need more."})

        elif action == "clarify":
            return {
                "sql": last_sql, "df": last_df, "sql_error": sql_error,
                "needs_clarification": True, "clarifying_question": parsed.get("clarifying_question", ""),
            }

        else:  # final_answer, or anything unrecognized — treat as "done with what we have"
            return {
                "sql": last_sql, "df": last_df, "sql_error": sql_error,
                "needs_clarification": False, "clarifying_question": None,
            }

    return {
        "sql": last_sql, "df": last_df, "sql_error": sql_error,
        "needs_clarification": False, "clarifying_question": None,
    }


# ---- quick check ----
_test = agent_step("What are the top 5 rows in the sales table by SALES?", get_schema())
print(_test["sql"])
print(_test["df"])

In [ ]:
def verify_answer(question: str, sql: str, df):
    """Second, independent model call whose only job is to sanity-check the first
    one — this catches confidently-wrong SQL (valid syntax, wrong logic) that a plain
    try/except around execution would never catch."""
    if sql is None:
        return False, "No query was executed."

    preview = df.head(20).to_json(orient="records") if df is not None else "[]"
    row_count = len(df) if df is not None else 0

    prompt = f"""Question: {question}
SQL used: {sql}
Rows returned: {row_count}
Preview: {preview}

Does this SQL genuinely answer the question? A zero-row result is only "correct" if the
question really has no matching data — not because a filter value might be misspelled or
the wrong case. Reply with ONLY: {{"is_correct": true or false, "reason": "one short sentence"}}"""

    response = ollama.chat(
        model=MODEL, format="json",
        messages=[
            {"role": "system", "content": "You are a strict SQL reviewer. Reply with ONLY valid JSON."},
            {"role": "user", "content": prompt},
        ],
    )
    try:
        parsed = json.loads(response["message"]["content"])
        return parsed["is_correct"], parsed.get("reason", "")
    except (json.JSONDecodeError, KeyError):
        return False, "Verifier didn't return valid JSON — treating as unverified."


def detect_ambiguous_question(question: str, schema: str):
    """Runs before the agent loop. Catches vague prompts ('how are we doing?') so
    the agent doesn't silently pick an interpretation and confidently answer the wrong
    thing."""
    prompt = f"""Schema:
{schema}

Question: "{question}"

Is this specific enough to turn into one exact SQL query? Reply with ONLY:
{{"is_ambiguous": true or false, "clarifying_question": "..." or null}}"""

    response = ollama.chat(
        model=MODEL, format="json",
        messages=[
            {"role": "system", "content": "You judge whether questions are answerable with SQL. Reply with ONLY valid JSON."},
            {"role": "user", "content": prompt},
        ],
    )
    try:
        parsed = json.loads(response["message"]["content"])
        return parsed["is_ambiguous"], parsed.get("clarifying_question")
    except (json.JSONDecodeError, KeyError):
        return False, None  # fail open — better to attempt an answer than block on a parsing glitch

In [ ]:
def pick_chart_type(df, question: str):
    # rule-based first (fast, free, deterministic) — only falls back to an LLM call
    # when the shape of the result is genuinely ambiguous
    if df is None or df.empty or df.shape == (1, 1):
        return {"chart_type": "none", "x": None, "y": None}

    columns = list(df.columns)
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    non_numeric_cols = [c for c in columns if c not in numeric_cols]
    q_lower = question.lower()

    target_metric = next((c for c in numeric_cols if c.lower() in q_lower), None)
    is_ranking = any(w in q_lower for w in ["top", "highest", "largest", "best", "rank"])

    if is_ranking and (target_metric or numeric_cols):
        y = target_metric or numeric_cols[0]
        label_cols = [c for c in non_numeric_cols if any(k in c.lower() for k in ["name", "code", "product"])]
        x = label_cols[0] if label_cols else (columns[0] if columns else None)
        return {"chart_type": "bar", "x": x, "y": y}

    if len(non_numeric_cols) == 1 and len(numeric_cols) == 1:
        return {"chart_type": "bar", "x": non_numeric_cols[0], "y": numeric_cols[0]}

    date_like = [c for c in non_numeric_cols if any(k in c.lower() for k in ["date", "time", "year"])]
    if date_like and numeric_cols:
        return {"chart_type": "line", "x": date_like[0], "y": target_metric or numeric_cols[0]}

    if len(numeric_cols) >= 2:
        y = target_metric or numeric_cols[1]
        x = next((c for c in numeric_cols if c != y), numeric_cols[0])
        return {"chart_type": "scatter", "x": x, "y": y}

    choice = call_llm([{"role": "user", "content":
        f"Question: {question}\nColumns: {columns}\nReply with exactly one word: bar, line, scatter, or none."}]).strip().lower()
    if choice not in ["bar", "line", "scatter", "none"]:
        choice = "none"
    x = non_numeric_cols[0] if non_numeric_cols else (columns[0] if columns else None)
    return {"chart_type": choice, "x": x, "y": target_metric or (numeric_cols[0] if numeric_cols else None)}


def make_chart(df, chart_type: str, x: str, y: str):
    if chart_type == "none" or df is None or df.empty or x not in df.columns or y not in df.columns:
        return None
    plot_fn = {"bar": px.bar, "line": px.line, "scatter": px.scatter}.get(chart_type)
    if plot_fn is None:
        return None
    fig = plot_fn(df, x=x, y=y)
    fig.update_layout(margin=dict(l=20, r=20, t=30, b=20))
    return fig


def write_narrative(question: str, df):
    # grounded in df only — the model writing this never sees the question without also
    # seeing the exact rows it has to base the answer on, so there's nothing to hallucinate from
    preview = df.head(20).to_json(orient="records")
    prompt = f"""Question: {question}
Data (up to 20 of {len(df)} rows): {preview}

Answer in 2-3 sentences, using only the numbers/values shown above. No outside knowledge, no speculation."""
    return call_llm([{"role": "user", "content": prompt}]).strip()

In [ ]:
def analyze(question: str) -> dict:
    """Wires it all together: ambiguity check -> agent loop -> verify -> retry
    (including on a query that ran fine but came back empty) -> chart + narrative.

    The empty-result handling here is the main fix over my first version: a zero-row
    result used to get reported straight to the user as "no data found," which was wrong
    more often than not — usually it meant the model picked a bad filter value, not that
    the data was actually missing. Now it's treated as an unverified answer and retried
    like any other mistake, and only becomes a final "no data" message if it survives
    retries, with the SQL shown so the user isn't just taking the agent's word for it.
    """
    start = time.time()
    schema = get_schema()

    is_ambiguous, clarifying_q = detect_ambiguous_question(question, schema)
    if is_ambiguous:
        return {
            "sql": None, "df": None, "chart": None, "narrative": None, "verified": None,
            "retries": 0, "latency": time.time() - start, "clarifying_question": clarifying_q,
        }

    result, verified, feedback, retries = None, False, None, 0

    while retries <= MAX_VERIFY_RETRIES:
        result = agent_step(question, schema, feedback=feedback)

        if result["needs_clarification"]:
            return {
                "sql": result["sql"], "df": result["df"], "chart": None, "narrative": None,
                "verified": None, "retries": retries, "latency": time.time() - start,
                "clarifying_question": result["clarifying_question"],
            }

        if result["sql_error"]:
            feedback, verified = f"Query kept failing: {result['sql_error']}", False
            retries += 1
            continue

        if result["df"] is not None and result["df"].empty:
            feedback = ("Query executed but returned zero rows. If a filter value might be "
                        "misspelled or wrong case, fix it and try again — only call final_answer "
                        "if you're confident no data matches.")
            verified = False
            retries += 1
            continue

        verified, feedback = verify_answer(question, result["sql"], result["df"])
        if verified:
            break
        retries += 1

    df = result["df"]

    if df is None:
        narrative = f"Couldn't get a working query after {retries} attempt(s). {result['sql_error'] or 'The agent finished without running one.'}"
        chart = None
    elif df.empty:
        narrative = f"Ran the query but got zero matching rows, even after double-checking filter values.\nSQL used: {result['sql']}"
        chart = None
    else:
        chart_choice = pick_chart_type(df, question)
        chart = make_chart(df, chart_choice["chart_type"], chart_choice["x"], chart_choice["y"])
        narrative = write_narrative(question, df)
        if not verified:
            narrative += " (Note: couldn't fully verify this one — sanity-check the numbers.)"

    return {
        "sql": result["sql"], "df": df, "chart": chart, "narrative": narrative, "verified": verified,
        "retries": retries, "latency": time.time() - start, "clarifying_question": None,
    }


# ---- quick check ----
out = analyze("What are the top 5 rows in the sales table by SALES?")
print("verified:", out["verified"], "| retries:", out["retries"], "| latency:", round(out["latency"], 2))
print(out["narrative"])
if out["chart"]:
    out["chart"].show()

In [ ]:
EVAL_QUESTIONS = [
    ("What is the total SALES in the sales table?", "answer"),
    ("What are the top 5 rows in the sales table by SALES?", "answer"),
    ("How many rows are in the sales table?", "answer"),
    ("How are we doing?", "clarify"),                                       # deliberately vague
    ("What is the total SALES for a product code that doesn't exist, XYZ999?", "empty"),
]


def run_eval():
    # not a full benchmark — a smoke test I run after touching the prompts or the retry
    # logic, so I catch a regression before a live demo instead of during one
    rows = []
    for question, expected in EVAL_QUESTIONS:
        out = analyze(question)
        if expected == "clarify":
            passed = out["clarifying_question"] is not None
        elif expected == "empty":
            passed = out["df"] is not None and out["df"].empty
        else:
            passed = out["df"] is not None and not out["df"].empty and out["verified"]
        rows.append({
            "question": question, "passed": passed, "verified": out["verified"],
            "retries": out["retries"], "latency_s": round(out["latency"], 2), "sql": out["sql"],
        })
    return pd.DataFrame(rows)


eval_results = run_eval()
print(f"{eval_results['passed'].sum()}/{len(eval_results)} passed")
eval_results

In [ ]:
def run_demo():
    print("=== Data Analyst Agent === (type 'quit' to stop)\n")
    while True:
        question = input("Your question: ").strip()
        if question.lower() in ("quit", "exit"):
            print("Goodbye.")
            break
        if not question:
            continue

        out = analyze(question)

        if out["clarifying_question"]:
            print(f"\nNeed more detail: {out['clarifying_question']}\n")
            continue

        print(f"\n{out['narrative']}")
        print(f"[verified: {out['verified']} | retries: {out['retries']} | {round(out['latency'], 2)}s]")
        if out["sql"]:
            print(f"SQL: {out['sql']}")
        if out["chart"]:
            out["chart"].show()
        if out["df"] is not None and not out["df"].empty:
            print(out["df"].head(10))
        print("-" * 60, "\n")


run_demo()